# synthkit — fully self-contained notebook

The ENTIRE library inlined below, generated from the
module sources by `scripts/build_notebook.py` — nothing
to install, no repo required: run top to bottom in any
JupyterLab (SageMaker Studio included).

Pipeline: plain-English spec -> deterministic planning
(blueprints ARE the ground truth) -> verified rendering
(code verifier around every LLM call, deterministic
fallback) -> exact sliced evaluation -> measured
experiments.

PHI posture: there is NO ingestion path anywhere below.
Distributions are specified, never fitted; nothing real
ever enters the generator.


## Library: The DataSpec — the reviewable contract

*Source of truth: `synthkit/spec.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S1: the DataSpec — the reviewable contract.

Plain English goes into the compiler; a DataSpec comes out; a human
reads and edits it BEFORE anything generates. Every downstream stage
(planner, renderers, evaluator) consumes only this object, so the
spec is the single place where "what should this corpus contain"
lives. JSON round-trip is exact: load(dump(spec)) == spec.

Design rules:
  - No ingestion path. Distributions are SPECIFIED, never fitted
    from data. Nothing real ever enters (the PHI posture is
    structural, not procedural).
  - Validation is loud and total: a spec either validates completely
    or raises SpecError listing every problem found — no partial
    acceptance, because the spec doubles as the evaluation ground
    truth's schema.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List, Optional

SPEC_VERSION = 1

FIELD_TYPES = ("int", "float", "str", "date", "categorical", "id",
               "person_name")
DISTRIBUTIONS = ("uniform", "normal", "lognormal", "categorical",
                 "date_range", "sequence")
DIFFICULTIES = ("easy", "medium", "hard")


class SpecError(ValueError):
    """Raised with EVERY validation problem, newline-joined."""


# ===================================================================
# Structured side
# ===================================================================

@dataclass
class Distribution:
    """How a structured field's values are drawn.

    kind        one of DISTRIBUTIONS
    params      kind-specific:
      uniform      {"low": x, "high": y}            (int/float)
      normal       {"mean": m, "stdev": s, "min"?: , "max"?: }
      lognormal    {"mu": m, "sigma": s, "min"?: , "max"?: }
      categorical  {"choices": ["a", ...], "weights"?: [0.5, ...]}
      date_range   {"start": "YYYY-MM-DD", "end": "YYYY-MM-DD"}
      sequence     {"prefix"?: "ENC-", "start"?: 1000}   (ids)
    """
    kind: str
    params: Dict[str, Any] = field(default_factory=dict)


@dataclass
class StructuredField:
    name: str
    ftype: str                       # one of FIELD_TYPES
    distribution: Distribution
    nullable_rate: float = 0.0       # fraction of docs where null


@dataclass
class CrossFieldRule:
    """V1 supports ordered pairs: `later` must exceed `earlier` by a
    delta drawn from [min_delta, max_delta] (days for dates, units
    for numerics). Enforced constructively at planning time — the
    planner samples `earlier` and a delta, never rejection-samples."""
    earlier: str
    later: str
    min_delta: float = 0.0
    max_delta: float = 30.0


# ===================================================================
# Unstructured side
# ===================================================================

@dataclass
class TargetElement:
    """A plantable fact the outside model SHOULD extract.

    element_id   stable key, becomes the ground-truth label
    description  what the fact is ("current medication with dose")
    phrasings    2+ surface realizations; the planner draws one per
                 document ("{value} 20mg daily", "pt continues
                 {value}"). '{value}' slots a sampled value when
                 value_source names a structured field or a
                 categorical pool.
    density      fraction of documents that contain this element
    difficulty   easy/medium/hard — a rendering hint (hard = buried,
                 abbreviated, or split across sentences) and an
                 evaluation slice
    value_source optional: structured field name or inline
                 {"choices": [...]} pool supplying '{value}'
    """
    element_id: str
    description: str
    phrasings: List[str]
    density: float = 1.0
    difficulty: str = "medium"
    value_source: Optional[Any] = None


@dataclass
class Distractor:
    """A plausible near-miss the model should NOT extract — the part
    of the spec that makes a test adversarial rather than a demo.
    Same shape as a target minus difficulty semantics."""
    distractor_id: str
    description: str
    phrasings: List[str]
    density: float = 0.5
    value_source: Optional[Any] = None


@dataclass
class StyleAxes:
    """Controlled variation axes; the planner draws one value per
    axis per document, and the evaluator slices results by them."""
    personas: List[str] = field(default_factory=lambda: ["neutral"])
    verbosity: List[str] = field(
        default_factory=lambda: ["terse", "moderate", "verbose"])
    abbreviation: List[str] = field(
        default_factory=lambda: ["none", "moderate", "heavy"])


@dataclass
class UnstructuredField:
    name: str
    note_type: str                   # "nursing progress note", ...
    target_elements: List[TargetElement] = field(default_factory=list)
    distractors: List[Distractor] = field(default_factory=list)
    style: StyleAxes = field(default_factory=StyleAxes)
    length_words: List[int] = field(
        default_factory=lambda: [80, 220])   # [min, max]


# ===================================================================
# The spec
# ===================================================================

@dataclass
class CorpusConfig:
    size: int = 50
    master_seed: int = 20260715


@dataclass
class DataSpec:
    title: str
    structured_fields: List[StructuredField] = field(default_factory=list)
    unstructured_fields: List[UnstructuredField] = field(default_factory=list)
    cross_field_rules: List[CrossFieldRule] = field(default_factory=list)
    corpus: CorpusConfig = field(default_factory=CorpusConfig)
    version: int = SPEC_VERSION

    # ---------------- serialization ----------------

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2, ensure_ascii=False)

    @staticmethod
    def from_json(text: str) -> "DataSpec":
        raw = json.loads(text)
        return DataSpec(
            title=raw.get("title", ""),
            structured_fields=[
                StructuredField(
                    name=f["name"], ftype=f["ftype"],
                    distribution=Distribution(**f["distribution"]),
                    nullable_rate=f.get("nullable_rate", 0.0),
                )
                for f in raw.get("structured_fields", [])
            ],
            unstructured_fields=[
                UnstructuredField(
                    name=u["name"], note_type=u.get("note_type", "note"),
                    target_elements=[
                        TargetElement(**t)
                        for t in u.get("target_elements", [])
                    ],
                    distractors=[
                        Distractor(**d) for d in u.get("distractors", [])
                    ],
                    style=StyleAxes(**u.get("style", {})),
                    length_words=u.get("length_words", [80, 220]),
                )
                for u in raw.get("unstructured_fields", [])
            ],
            cross_field_rules=[
                CrossFieldRule(**r)
                for r in raw.get("cross_field_rules", [])
            ],
            corpus=CorpusConfig(**raw.get("corpus", {})),
            version=raw.get("version", SPEC_VERSION),
        )

    # ---------------- validation ----------------

    def validate(self) -> None:
        """Raises SpecError with EVERY problem, or returns quietly."""
        problems: List[str] = []
        names = set()

        for f in self.structured_fields:
            where = "structured field `{}`".format(f.name or "?")
            if not f.name:
                problems.append("structured field with empty name")
            elif f.name in names:
                problems.append("duplicate field name `{}`".format(f.name))
            names.add(f.name)
            if f.ftype not in FIELD_TYPES:
                problems.append("{}: unknown type `{}` (know: {})".format(
                    where, f.ftype, ", ".join(FIELD_TYPES)))
            d = f.distribution
            if d.kind not in DISTRIBUTIONS:
                problems.append("{}: unknown distribution `{}`".format(
                    where, d.kind))
            elif d.kind == "categorical":
                choices = d.params.get("choices") or []
                if not choices:
                    problems.append("{}: categorical needs choices".format(
                        where))
                weights = d.params.get("weights")
                if weights is not None and len(weights) != len(choices):
                    problems.append(
                        "{}: {} weights for {} choices".format(
                            where, len(weights), len(choices)))
            elif d.kind == "uniform":
                if not {"low", "high"} <= set(d.params):
                    problems.append("{}: uniform needs low/high".format(
                        where))
            elif d.kind == "normal":
                if not {"mean", "stdev"} <= set(d.params):
                    problems.append("{}: normal needs mean/stdev".format(
                        where))
            elif d.kind == "lognormal":
                if not {"mu", "sigma"} <= set(d.params):
                    problems.append("{}: lognormal needs mu/sigma".format(
                        where))
            elif d.kind == "date_range":
                if not {"start", "end"} <= set(d.params):
                    problems.append("{}: date_range needs start/end".format(
                        where))
            if not (0.0 <= f.nullable_rate <= 1.0):
                problems.append("{}: nullable_rate outside [0,1]".format(
                    where))

        for u in self.unstructured_fields:
            where = "unstructured field `{}`".format(u.name or "?")
            if not u.name:
                problems.append("unstructured field with empty name")
            elif u.name in names:
                problems.append("duplicate field name `{}`".format(u.name))
            names.add(u.name)
            if not u.target_elements:
                problems.append("{}: no target elements — nothing to "
                                "test".format(where))
            eids = set()
            for t in u.target_elements:
                ew = "{} element `{}`".format(where, t.element_id or "?")
                if not t.element_id:
                    problems.append("{}: element with empty id".format(
                        where))
                elif t.element_id in eids:
                    problems.append("duplicate element id `{}`".format(
                        t.element_id))
                eids.add(t.element_id)
                if len(t.phrasings) < 1:
                    problems.append("{}: needs at least 1 phrasing".format(
                        ew))
                if not (0.0 < t.density <= 1.0):
                    problems.append("{}: density outside (0,1]".format(ew))
                if t.difficulty not in DIFFICULTIES:
                    problems.append("{}: difficulty `{}` unknown".format(
                        ew, t.difficulty))
                problems.extend(_check_value_source(
                    t.value_source, t.phrasings, names, ew))
            for dtr in u.distractors:
                dw = "{} distractor `{}`".format(
                    where, dtr.distractor_id or "?")
                if dtr.distractor_id in eids:
                    problems.append(
                        "{}: id collides with a target element".format(dw))
                if not dtr.phrasings:
                    problems.append("{}: needs phrasings".format(dw))
                problems.extend(_check_value_source(
                    dtr.value_source, dtr.phrasings, names, dw))
            if (len(u.length_words) != 2
                    or u.length_words[0] > u.length_words[1]
                    or u.length_words[0] < 10):
                problems.append("{}: length_words must be [min>=10, "
                                "max>=min]".format(where))

        rule_fields = {f.name for f in self.structured_fields}
        for r in self.cross_field_rules:
            for side in (r.earlier, r.later):
                if side not in rule_fields:
                    problems.append(
                        "cross-field rule references unknown field "
                        "`{}`".format(side))
            if r.min_delta > r.max_delta:
                problems.append(
                    "cross-field rule {}<{}: min_delta > max_delta".format(
                        r.earlier, r.later))

        if self.corpus.size < 1:
            problems.append("corpus size must be >= 1")
        if not self.title:
            problems.append("spec needs a title")

        if problems:
            raise SpecError("\n".join(problems))


def _check_value_source(source: Any, phrasings: List[str],
                        field_names: set, where: str) -> List[str]:
    out: List[str] = []
    uses_value = any("{value}" in p for p in phrasings)
    if source is None:
        if uses_value:
            out.append("{}: phrasings use {{value}} but no "
                       "value_source".format(where))
        return out
    if isinstance(source, str):
        if source not in field_names:
            out.append("{}: value_source names unknown field "
                       "`{}`".format(where, source))
    elif isinstance(source, dict):
        if not source.get("choices"):
            out.append("{}: inline value_source needs choices".format(
                where))
    else:
        out.append("{}: value_source must be a field name or "
                   "{{'choices': [...]}}".format(where))
    if not uses_value:
        out.append("{}: value_source given but no phrasing uses "
                   "{{value}}".format(where))
    return out


## Library: LLM backends (Ollama / Anthropic / Bedrock) and the spec compiler

*Source of truth: `synthkit/compiler.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S1: backends and the SpecCompiler.

The LLMBackend protocol is the deployment seam: development runs
Ollama on the M3; Keck runs Bedrock or the Anthropic API. Every
LLM-touching stage takes a backend instance, so the move is a config
change, not a port.

The compiler is LLM-ASSISTED, not LLM-trusted: plain English goes
in, a draft DataSpec comes out, validation runs, and the human
reviews the JSON before anything generates. A compile that fails
validation returns the problems alongside the draft — the fix loop
is human-in-the-middle by design.

Python 3.8 compatible. Stdlib only (backends import their SDKs
lazily so the library works wherever at least one is available).
"""
from __future__ import annotations

import json
import logging
import re
from dataclasses import dataclass
from typing import Any, Optional, Tuple

pass  # intra-package import inlined above

log = logging.getLogger(__name__)


# ===================================================================
# Backends
# ===================================================================

class LLMBackend:
    """Protocol: complete(prompt, system, max_tokens, temperature)
    -> str. Adapters raise BackendError on hard failure."""

    name = "base"

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        raise NotImplementedError


class BackendError(RuntimeError):
    pass


class OllamaBackend(LLMBackend):
    """Local models via the Ollama HTTP API (development default)."""

    name = "ollama"

    def __init__(self, model: str = "mistral-small3.1",
                 host: str = "http://localhost:11434",
                 timeout_s: float = 300.0):
        self.model = model
        self.host = host.rstrip("/")
        self.timeout_s = timeout_s

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        import urllib.request
        body = json.dumps({
            "model": self.model,
            "prompt": prompt,
            "system": system,
            "stream": False,
            "options": {"temperature": temperature,
                        "num_predict": max_tokens},
        }).encode("utf-8")
        req = urllib.request.Request(
            self.host + "/api/generate", data=body,
            headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req,
                                        timeout=self.timeout_s) as r:
                return json.loads(r.read().decode("utf-8")).get(
                    "response", "")
        except Exception as e:
            raise BackendError("ollama call failed: {}".format(e))


class AnthropicBackend(LLMBackend):
    """Anthropic API (lazy SDK import)."""

    name = "anthropic"

    def __init__(self, model: str = "claude-sonnet-4-6",
                 api_key: Optional[str] = None):
        self.model = model
        self.api_key = api_key

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        try:
            import anthropic
        except ImportError:
            raise BackendError("anthropic SDK not installed")
        client = anthropic.Anthropic(api_key=self.api_key) \
            if self.api_key else anthropic.Anthropic()
        try:
            msg = client.messages.create(
                model=self.model,
                max_tokens=max_tokens,
                temperature=temperature,
                system=system or None,
                messages=[{"role": "user", "content": prompt}],
            )
            return "".join(
                b.text for b in msg.content
                if getattr(b, "type", "") == "text")
        except Exception as e:
            raise BackendError("anthropic call failed: {}".format(e))


class HFLocalBackend(LLMBackend):
    """IN-PROCESS open-source model via Hugging Face transformers —
    generation happens inside the notebook's own Python process, no
    external server or service.

        pip install transformers torch accelerate

    Weights download from the Hugging Face hub on first use (or pass
    a local path / S3-synced directory as model_id for air-gapped
    environments). Small instruct models are the sweet spot for
    synthkit's short verified renders:

        HFLocalBackend("Qwen/Qwen2.5-1.5B-Instruct")   # ~3GB
        HFLocalBackend("Qwen/Qwen2.5-0.5B-Instruct")   # ~1GB, CPU-ok

    The renderer's verifier + retry + fallback wrap this like any
    backend: a weak model degrades measurably, never breaks the
    corpus. `pipeline` is injectable for tests.
    """

    def __init__(self, model_id: str = "Qwen/Qwen2.5-1.5B-Instruct",
                 device_map: str = "auto",
                 pipeline: Any = None):
        self.model_id = model_id
        self.name = "hf-local/" + model_id
        self._device_map = device_map
        self._pipe = pipeline

    def _ensure_pipe(self) -> Any:
        if self._pipe is None:
            try:
                from transformers import pipeline as hf_pipeline
            except ImportError:
                raise BackendError(
                    "transformers is not installed — run: pip "
                    "install transformers torch accelerate")
            try:
                self._pipe = hf_pipeline(
                    "text-generation", model=self.model_id,
                    device_map=self._device_map)
            except Exception as e:
                raise BackendError(
                    "could not load {}: {}".format(self.model_id, e))
        return self._pipe

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        pipe = self._ensure_pipe()
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        kwargs = {"max_new_tokens": max_tokens,
                  "do_sample": temperature > 0,
                  "temperature": max(temperature, 0.01),
                  "return_full_text": False}
        try:
            out = pipe(messages, **kwargs)
        except (TypeError, ValueError):
            text_in = (system + "\n\n" + prompt) if system else prompt
            try:
                out = pipe(text_in, **kwargs)
            except Exception as e:
                raise BackendError("hf-local generation failed: "
                                   "{}".format(e))
        except Exception as e:
            raise BackendError("hf-local generation failed: "
                               "{}".format(e))
        try:
            text = out[0]["generated_text"]
            if isinstance(text, list):
                text = text[-1].get("content", "")
            return str(text)
        except (KeyError, IndexError, TypeError, AttributeError) as e:
            raise BackendError(
                "hf-local returned an unexpected shape: {}".format(e))


class BedrockBackend(LLMBackend):
    """AWS Bedrock via boto3 (the Keck deployment path).
    Anthropic-on-Bedrock message format."""

    name = "bedrock"

    def __init__(self, model_id: str, region: str = "us-west-2"):
        self.model_id = model_id
        self.region = region

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        try:
            import boto3
        except ImportError:
            raise BackendError("boto3 not installed")
        client = boto3.client("bedrock-runtime",
                              region_name=self.region)
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}],
        }
        if system:
            body["system"] = system
        try:
            resp = client.invoke_model(
                modelId=self.model_id,
                body=json.dumps(body))
            parsed = json.loads(resp["body"].read())
            return "".join(
                b.get("text", "") for b in parsed.get("content", []))
        except Exception as e:
            raise BackendError("bedrock call failed: {}".format(e))


# ===================================================================
# The compiler
# ===================================================================

_COMPILER_SYSTEM = """You compile a plain-English description of a \
synthetic dataset into a strict JSON DataSpec. Output ONLY the JSON \
object, no prose, no markdown fences, starting with '{'.

Schema (all keys required unless noted):
{
  "title": "short name for the dataset",
  "structured_fields": [
    {"name": "field_name",
     "ftype": "int|float|str|date|categorical|id|person_name",
     "distribution": {"kind": "uniform|normal|lognormal|categorical|date_range|sequence",
                      "params": { ... kind-specific ... }},
     "nullable_rate": 0.0}
  ],
  "unstructured_fields": [
    {"name": "field_name", "note_type": "what kind of note this is",
     "target_elements": [
       {"element_id": "snake_case_id",
        "description": "the fact an extractor should find",
        "phrasings": ["two or more surface forms, use {value} where a sampled value goes"],
        "density": 0.8, "difficulty": "easy|medium|hard",
        "value_source": "structured_field_name OR {\\"choices\\": [..]} OR null"}
     ],
     "distractors": [
       {"distractor_id": "snake_case_id",
        "description": "plausible near-miss that should NOT be extracted",
        "phrasings": ["..."], "density": 0.5, "value_source": null}
     ],
     "style": {"personas": ["..."], "verbosity": ["terse","moderate","verbose"],
               "abbreviation": ["none","moderate","heavy"]},
     "length_words": [80, 220]}
  ],
  "cross_field_rules": [
    {"earlier": "field_a", "later": "field_b",
     "min_delta": 0, "max_delta": 30}
  ],
  "corpus": {"size": 50, "master_seed": 20260715}
}

Distribution params: uniform {"low","high"}; normal {"mean","stdev",
optional "min","max"}; lognormal {"mu","sigma",optional "min","max"};
categorical {"choices",optional "weights"}; date_range {"start","end"
as YYYY-MM-DD}; sequence {"prefix","start"} for ids.

Rules:
- Every target element gets >=2 phrasings with genuinely different
  surface structure.
- Include distractors: plausible near-misses of the targets.
- densities in (0,1]; difficulties spread across easy/medium/hard.
- NEVER invent real-sounding institutions, patients, or providers;
  values are generic and synthetic.
- Sizes and seeds: honor the user's numbers; default size 50."""


@dataclass
class CompileResult:
    spec: Optional[DataSpec]
    raw_json: str
    problems: Optional[str]      # None when the spec validates

    @property
    def ok(self) -> bool:
        return self.spec is not None and self.problems is None


def _extract_json(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return m.group(0) if m else None


def compile_spec(description: str, backend: LLMBackend,
                 retries: int = 1) -> CompileResult:
    """Plain English -> validated DataSpec (or draft + problems).
    On validation failure, one repair round feeds the problems back
    to the model; after that, the human takes over — by design."""
    prompt = ("Dataset description:\n\n{}\n\n=== END DESCRIPTION ===\n"
              "Now output ONLY the DataSpec JSON object, starting "
              "with '{{'.".format(description.strip()))
    last_raw = ""
    last_problems = "no output produced"
    for attempt in range(retries + 1):
        raw = backend.complete(prompt, system=_COMPILER_SYSTEM,
                               max_tokens=3000, temperature=0.3)
        json_str = _extract_json(raw)
        if json_str is None:
            last_raw, last_problems = raw or "", "no JSON object in output"
        else:
            last_raw = json_str
            try:
                spec = DataSpec.from_json(json_str)
            except (json.JSONDecodeError, TypeError, KeyError) as e:
                last_problems = "JSON did not fit the schema: {}".format(e)
            else:
                try:
                    spec.validate()
                    return CompileResult(spec=spec, raw_json=json_str,
                                         problems=None)
                except SpecError as e:
                    last_problems = str(e)
                    log.info("compile attempt %d: %d validation "
                             "problem(s)", attempt + 1,
                             last_problems.count("\n") + 1)
        if attempt < retries:
            prompt = (
                "Dataset description:\n\n{}\n\n"
                "Your previous DataSpec had these problems:\n{}\n\n"
                "Output the CORRECTED DataSpec JSON only, starting "
                "with '{{'.".format(description.strip(), last_problems)
            )
    try:
        draft = DataSpec.from_json(last_raw)
    except Exception:
        draft = None
    return CompileResult(spec=draft, raw_json=last_raw,
                         problems=last_problems)


## Library: The Planner — deterministic blueprints, ground truth first

*Source of truth: `synthkit/planner.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S2: the Planner — spec in, blueprints out, seeded.

One blueprint per document: which target elements it contains (per
their densities), the sampled values, the drawn style, which
distractors are present, and every structured field's value. The
blueprint IS the ground truth — labels precede the data, so no
labeling step ever exists.

Determinism contract: the same spec + master_seed produces the same
blueprints byte for byte, on any machine, forever. Per-document RNGs
derive from (master_seed, doc_index) so corpus size changes never
reshuffle existing documents.

Cross-field rules are enforced CONSTRUCTIVELY: the planner samples
the `earlier` field and a delta, never rejection-samples — a spec
that validates always plans.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import hashlib
import json
import math
import random
from dataclasses import asdict, dataclass, field
from datetime import date, timedelta
from typing import Any, Dict, List, Optional

pass  # intra-package import inlined above

FIRST_NAMES = (
    "Avery", "Jordan", "Riley", "Quinn", "Morgan", "Casey", "Rowan",
    "Skyler", "Emerson", "Hayden", "Reese", "Dakota", "Finley",
    "Sage", "Marlowe", "Ellis",
)
LAST_NAMES = (
    "Calloway", "Mercer", "Ashford", "Brennan", "Voss", "Hale",
    "Winslow", "Marsh", "Keating", "Solano", "Iverson", "Trent",
    "Fontaine", "Barlow", "Quimby", "Renner",
)


@dataclass
class PlannedElement:
    element_id: str
    phrasing: str            # chosen phrasing with {value} resolved
    value: Optional[str]     # the resolved value ('' family), if any
    difficulty: str


@dataclass
class PlannedDistractor:
    distractor_id: str
    phrasing: str
    value: Optional[str]


@dataclass
class PlannedNote:
    field_name: str
    note_type: str
    style: Dict[str, str]            # axis -> drawn value
    length_words: int
    elements: List[PlannedElement] = field(default_factory=list)
    distractors: List[PlannedDistractor] = field(default_factory=list)


@dataclass
class Blueprint:
    doc_id: str
    doc_index: int
    seed: int
    structured: Dict[str, Any] = field(default_factory=dict)
    notes: List[PlannedNote] = field(default_factory=list)

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2, ensure_ascii=False,
                          default=str)


def _doc_seed(master_seed: int, doc_index: int) -> int:
    """Stable per-document seed independent of corpus size."""
    h = hashlib.sha256("{}:{}".format(master_seed, doc_index)
                       .encode("utf-8")).hexdigest()
    return int(h[:12], 16)


def _sample_distribution(rng: random.Random, ftype: str,
                         d: Distribution) -> Any:
    p = d.params
    if d.kind == "uniform":
        v = rng.uniform(float(p["low"]), float(p["high"]))
        return int(round(v)) if ftype == "int" else round(v, 3)
    if d.kind == "normal":
        v = rng.gauss(float(p["mean"]), float(p["stdev"]))
        v = _clamp(v, p.get("min"), p.get("max"))
        return int(round(v)) if ftype == "int" else round(v, 3)
    if d.kind == "lognormal":
        v = rng.lognormvariate(float(p["mu"]), float(p["sigma"]))
        v = _clamp(v, p.get("min"), p.get("max"))
        return int(round(v)) if ftype == "int" else round(v, 3)
    if d.kind == "categorical":
        choices = p["choices"]
        weights = p.get("weights")
        return rng.choices(choices, weights=weights, k=1)[0]
    if d.kind == "date_range":
        start = date.fromisoformat(p["start"])
        end = date.fromisoformat(p["end"])
        span = max((end - start).days, 0)
        return (start + timedelta(days=rng.randint(0, span))).isoformat()
    if d.kind == "sequence":
        # Resolved by the caller (needs doc_index, not randomness).
        return None
    raise SpecError("unsupported distribution kind: {}".format(d.kind))


def _clamp(v: float, lo: Optional[Any], hi: Optional[Any]) -> float:
    if lo is not None:
        v = max(float(lo), v)
    if hi is not None:
        v = min(float(hi), v)
    return v


def _person_name(rng: random.Random) -> str:
    return "{} {}".format(rng.choice(FIRST_NAMES),
                          rng.choice(LAST_NAMES))


def _resolve_value(rng: random.Random, source: Any,
                   structured: Dict[str, Any]) -> Optional[str]:
    if source is None:
        return None
    if isinstance(source, str):
        return str(structured.get(source, ""))
    if isinstance(source, dict):
        return str(rng.choice(source["choices"]))
    return None


def plan_document(spec: DataSpec, doc_index: int) -> Blueprint:
    """One document's blueprint, fully determined by (spec, index)."""
    seed = _doc_seed(spec.corpus.master_seed, doc_index)
    rng = random.Random(seed)
    bp = Blueprint(
        doc_id="doc_{:05d}".format(doc_index),
        doc_index=doc_index,
        seed=seed,
    )

    # ---- structured fields (rule-constrained pairs handled after) ----
    ruled_later = {r.later: r for r in spec.cross_field_rules}
    for f in spec.structured_fields:
        if f.name in ruled_later:
            continue
        if f.nullable_rate and rng.random() < f.nullable_rate:
            bp.structured[f.name] = None
            continue
        if f.distribution.kind == "sequence":
            p = f.distribution.params
            bp.structured[f.name] = "{}{}".format(
                p.get("prefix", ""), int(p.get("start", 1)) + doc_index)
        elif f.ftype == "person_name":
            bp.structured[f.name] = _person_name(rng)
        else:
            bp.structured[f.name] = _sample_distribution(
                rng, f.ftype, f.distribution)

    for r in spec.cross_field_rules:
        earlier_val = bp.structured.get(r.earlier)
        f = next((x for x in spec.structured_fields
                  if x.name == r.later), None)
        if f is None or earlier_val is None:
            continue
        delta = rng.uniform(r.min_delta, r.max_delta)
        if f.ftype == "date":
            base = date.fromisoformat(str(earlier_val))
            bp.structured[f.name] = (
                base + timedelta(days=int(math.ceil(delta)))
            ).isoformat()
        else:
            v = float(earlier_val) + delta
            bp.structured[f.name] = (
                int(round(v)) if f.ftype == "int" else round(v, 3))

    # ---- unstructured notes ----
    for u in spec.unstructured_fields:
        note = PlannedNote(
            field_name=u.name,
            note_type=u.note_type,
            style={
                "persona": rng.choice(u.style.personas),
                "verbosity": rng.choice(u.style.verbosity),
                "abbreviation": rng.choice(u.style.abbreviation),
            },
            length_words=rng.randint(u.length_words[0],
                                     u.length_words[1]),
        )
        for t in u.target_elements:
            if rng.random() >= t.density:
                continue
            value = _resolve_value(rng, t.value_source, bp.structured)
            phrasing = rng.choice(t.phrasings)
            if value is not None:
                phrasing = phrasing.replace("{value}", value)
            note.elements.append(PlannedElement(
                element_id=t.element_id,
                phrasing=phrasing,
                value=value,
                difficulty=t.difficulty,
            ))
        for dtr in u.distractors:
            if rng.random() >= dtr.density:
                continue
            value = _resolve_value(rng, dtr.value_source, bp.structured)
            phrasing = rng.choice(dtr.phrasings)
            if value is not None:
                phrasing = phrasing.replace("{value}", value)
            note.distractors.append(PlannedDistractor(
                distractor_id=dtr.distractor_id,
                phrasing=phrasing,
                value=value,
            ))
        bp.notes.append(note)

    return bp


def plan_corpus(spec: DataSpec) -> List[Blueprint]:
    """Validates, then plans every document. Deterministic."""
    spec.validate()
    return [plan_document(spec, i) for i in range(spec.corpus.size)]


def corpus_stats(blueprints: List[Blueprint]) -> Dict[str, Any]:
    """Planned-corpus telemetry: realized element densities and style
    distribution — the pre-flight check that the corpus you are about
    to render actually exercises what the spec intended."""
    n = len(blueprints) or 1
    element_counts: Dict[str, int] = {}
    distractor_counts: Dict[str, int] = {}
    style_counts: Dict[str, Dict[str, int]] = {}
    for bp in blueprints:
        for note in bp.notes:
            for el in note.elements:
                element_counts[el.element_id] = (
                    element_counts.get(el.element_id, 0) + 1)
            for d in note.distractors:
                distractor_counts[d.distractor_id] = (
                    distractor_counts.get(d.distractor_id, 0) + 1)
            for axis, val in note.style.items():
                style_counts.setdefault(axis, {})
                style_counts[axis][val] = (
                    style_counts[axis].get(val, 0) + 1)
    return {
        "documents": len(blueprints),
        "element_density": {
            k: round(v / n, 3) for k, v in sorted(element_counts.items())
        },
        "distractor_density": {
            k: round(v / n, 3)
            for k, v in sorted(distractor_counts.items())
        },
        "style_distribution": style_counts,
    }


## Library: The Renderer — verified generation with targeted retries and fallback

*Source of truth: `synthkit/renderer.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S3: the Renderer — blueprints become documents, verified.

The LLM's one job here is prose: given a blueprint's planted
elements, distractors, style draws, and structured context, write a
natural note of the right register. Everything around that call is
code:

  VERIFIER (code, not LLM)
    - every planted element's needle (its value, or its phrasing
      when valueless) must appear in the text, normalized
    - every planted distractor's needle must appear (distractors are
      real content — the trap only works if it is present)
    - no OMITTED element may sneak in: for omitted elements with a
      choices pool, none of the pool values may appear — this is
      what keeps "absence is ground truth" true after an LLM has
      touched the data

  RETRY loop (default 3 attempts): failures are named verbatim in
  the retry prompt ("MUST include exactly: ...", "MUST NOT
  mention: ..."). Small window, explicit instruction — the only
  regime local models honor.

  FALLBACK: after retries, the deterministic stub renderer produces
  the document — valid by construction, flagged in the report. A
  corpus render therefore CANNOT fail; it can only report how much
  fallback it needed, which is itself renderer-quality telemetry.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import logging
import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above

log = logging.getLogger(__name__)

RENDER_ATTEMPTS = 3


def _norm(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip().lower()


def _needle(value: Optional[str], phrasing: str) -> str:
    return _norm(value) if value else _norm(phrasing)


# ===================================================================
# Verification
# ===================================================================

@dataclass
class VerifyResult:
    ok: bool
    missing: List[str] = field(default_factory=list)     # needles absent
    forbidden: List[str] = field(default_factory=list)   # leaked values


def _forbidden_values(spec_field: UnstructuredField,
                      note: PlannedNote) -> List[str]:
    """Pool values of OMITTED elements — their appearance would
    corrupt the absence side of the ground truth."""
    present = {el.element_id for el in note.elements}
    out: List[str] = []
    for t in spec_field.target_elements:
        if t.element_id in present:
            continue
        if isinstance(t.value_source, dict):
            out.extend(str(c) for c in
                       t.value_source.get("choices", []))
    # Values planted in THIS note are never forbidden even if they
    # also sit in an omitted element's pool.
    planted = {_norm(el.value) for el in note.elements if el.value}
    planted |= {_norm(d.value) for d in note.distractors if d.value}
    return [v for v in out if _norm(v) not in planted]


def verify_note(text: str, note: PlannedNote,
                spec_field: UnstructuredField) -> VerifyResult:
    body = _norm(text)
    missing: List[str] = []
    for el in note.elements:
        n = _needle(el.value, el.phrasing)
        if n and n not in body:
            missing.append(el.value or el.phrasing)
    for d in note.distractors:
        n = _needle(d.value, d.phrasing)
        if n and n not in body:
            missing.append(d.value or d.phrasing)
    forbidden = [v for v in _forbidden_values(spec_field, note)
                 if _norm(v) in body]
    return VerifyResult(ok=not missing and not forbidden,
                        missing=missing, forbidden=forbidden)


# ===================================================================
# Prompting
# ===================================================================

_RENDER_SYSTEM = (
    "You write realistic synthetic documents for testing information "
    "extraction systems. You will receive a note type, a style, "
    "structured context, and REQUIRED CONTENT items. Write ONE "
    "document only — no preamble, no markdown fences, no "
    "explanations.\n\n"
    "Hard rules:\n"
    "- Every REQUIRED CONTENT item's key text must appear VERBATIM "
    "(you may write naturally around it, but the exact text must be "
    "present).\n"
    "- Never mention anything listed under DO NOT MENTION.\n"
    "- Everything is synthetic; never invent real institutions or "
    "real people beyond the names given.\n"
    "- Match the requested persona, verbosity, abbreviation level, "
    "and approximate length."
)


def _style_line(note: PlannedNote) -> str:
    return ("persona: {persona}; verbosity: {verbosity}; "
            "abbreviations: {abbreviation}").format(**note.style)


def _render_prompt(bp: Blueprint, note: PlannedNote,
                   spec_field: UnstructuredField,
                   extra_missing: Optional[List[str]] = None,
                   extra_forbidden: Optional[List[str]] = None) -> str:
    required = [el.phrasing for el in note.elements]
    required += [d.phrasing for d in note.distractors]
    ctx = ", ".join("{}={}".format(k, v)
                    for k, v in sorted(bp.structured.items())
                    if v is not None)
    lines = [
        "Note type: {}".format(note.note_type),
        "Style: {}".format(_style_line(note)),
        "Approximate length: {} words".format(note.length_words),
        "Structured context (weave in naturally where sensible): "
        + ctx,
        "",
        "REQUIRED CONTENT (each key text VERBATIM):",
    ]
    lines += ["- {}".format(r) for r in required]
    forbidden = _forbidden_values(spec_field, note)
    if forbidden or extra_forbidden:
        lines.append("")
        lines.append("DO NOT MENTION:")
        lines += ["- {}".format(f)
                  for f in sorted(set(forbidden)
                                  | set(extra_forbidden or []))]
    if extra_missing:
        lines.append("")
        lines.append("YOUR PREVIOUS ATTEMPT OMITTED these — include "
                      "each VERBATIM this time:")
        lines += ["- {}".format(m) for m in extra_missing]
    if extra_forbidden:
        lines.append("")
        lines.append("YOUR PREVIOUS ATTEMPT MENTIONED these "
                      "forbidden items — write the document WITHOUT "
                      "any of them:")
        lines += ["- {}".format(f) for f in extra_forbidden]
    lines.append("")
    lines.append("=== WRITE THE DOCUMENT NOW ===")
    return "\n".join(lines)


# ===================================================================
# The stub renderer — deterministic, valid by construction
# ===================================================================

def render_stub(bp: Blueprint, note: PlannedNote) -> str:
    parts = []
    name = bp.structured.get("patient_name")
    ident = next((v for k, v in sorted(bp.structured.items())
                  if isinstance(v, str) and "-" in str(v)), None)
    parts.append("{} regarding {}{}.".format(
        note.note_type.capitalize(),
        name or "the patient",
        " ({})".format(ident) if ident else ""))
    parts.append("Documented by the {} ({}).".format(
        note.style.get("persona", "author"),
        _style_line(note)))
    for el in note.elements:
        parts.append(el.phrasing.rstrip(".") + ".")
    for d in note.distractors:
        parts.append(d.phrasing.rstrip(".") + ".")
    parts.append("Plan reviewed; reassess at next contact.")
    return " ".join(parts)


# ===================================================================
# Rendering
# ===================================================================

@dataclass
class RenderResult:
    doc_id: str
    field_name: str
    text: str
    attempts: int
    used_fallback: bool
    problems: List[str] = field(default_factory=list)


@dataclass
class RenderReport:
    documents: int = 0
    notes_rendered: int = 0
    first_try: int = 0
    retried: int = 0
    fallbacks: int = 0
    miss_counts: Dict[str, int] = field(default_factory=dict)

    def format_text(self) -> str:
        lines = [
            "RENDER REPORT: {} note(s) across {} document(s)".format(
                self.notes_rendered, self.documents),
            "  verified first try: {}".format(self.first_try),
            "  verified after retry: {}".format(self.retried),
            "  deterministic fallback: {}".format(self.fallbacks),
        ]
        if self.miss_counts:
            lines.append("  renderer misses by needle (pre-retry):")
            top = sorted(self.miss_counts.items(),
                         key=lambda kv: -kv[1])[:8]
            lines += ["    {:<44} {}".format(k[:44], v)
                      for k, v in top]
        return "\n".join(lines)


def render_note(bp: Blueprint, note: PlannedNote,
                spec_field: UnstructuredField,
                backend: LLMBackend,
                attempts: int = RENDER_ATTEMPTS,
                report: Optional[RenderReport] = None,
                ) -> RenderResult:
    """One note: render -> verify -> targeted retry -> fallback."""
    extra_missing: List[str] = []
    extra_forbidden: List[str] = []
    problems: List[str] = []
    for attempt in range(1, attempts + 1):
        prompt = _render_prompt(bp, note, spec_field,
                                extra_missing or None,
                                extra_forbidden or None)
        try:
            text = backend.complete(
                prompt, system=_RENDER_SYSTEM,
                max_tokens=max(400, note.length_words * 3),
                temperature=0.7 if attempt == 1 else 0.4,
            )
        except BackendError as e:
            problems.append("attempt {}: backend error: {}".format(
                attempt, e))
            continue
        v = verify_note(text or "", note, spec_field)
        if v.ok:
            return RenderResult(
                doc_id=bp.doc_id, field_name=note.field_name,
                text=text.strip(), attempts=attempt,
                used_fallback=False, problems=problems)
        problems.append("attempt {}: missing={} forbidden={}".format(
            attempt, v.missing, v.forbidden))
        if report is not None:
            for m in v.missing:
                report.miss_counts[m] = (
                    report.miss_counts.get(m, 0) + 1)
        extra_missing = v.missing
        extra_forbidden = v.forbidden
        log.info("S3: %s/%s attempt %d failed verification "
                 "(%d missing, %d forbidden)", bp.doc_id,
                 note.field_name, attempt, len(v.missing),
                 len(v.forbidden))
    return RenderResult(
        doc_id=bp.doc_id, field_name=note.field_name,
        text=render_stub(bp, note), attempts=attempts,
        used_fallback=True, problems=problems)


def render_corpus(spec: DataSpec, blueprints: List[Blueprint],
                  backend: LLMBackend,
                  attempts: int = RENDER_ATTEMPTS,
                  ) -> Tuple[Dict[str, str], RenderReport]:
    """Every blueprint's notes rendered and verified. Returns
    ({doc_id: text}, report). Multi-note documents concatenate with
    field headers. CANNOT fail — only degrade, measurably."""
    fields = {u.name: u for u in spec.unstructured_fields}
    report = RenderReport(documents=len(blueprints))
    documents: Dict[str, str] = {}
    for bp in blueprints:
        chunks: List[str] = []
        for note in bp.notes:
            spec_field = fields[note.field_name]
            r = render_note(bp, note, spec_field, backend,
                            attempts=attempts, report=report)
            report.notes_rendered += 1
            if r.used_fallback:
                report.fallbacks += 1
            elif r.attempts == 1:
                report.first_try += 1
            else:
                report.retried += 1
            if len(bp.notes) > 1:
                chunks.append("[{}]\n{}".format(note.field_name,
                                                r.text))
            else:
                chunks.append(r.text)
        documents[bp.doc_id] = "\n\n".join(chunks)
    return documents, report


## Library: Corpus I/O — the reproducible, integrity-checked run artifact

*Source of truth: `synthkit/corpus_io.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S3: corpus I/O — a run is a reproducible, auditable
object you can hand to a colleague.

Layout of corpus/<run_id>/:
    manifest.json    spec (verbatim), master seed, backend name,
                     render stats, sha256 of every artifact
    docs/<doc_id>.txt
    truth/<doc_id>.json      the blueprint — labels precede data
    render_report.json

load_corpus verifies every hash on the way in: a corpus that fails
integrity refuses to load, because an edited document silently
diverging from its blueprint would poison every evaluation after it.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import hashlib
import json
from dataclasses import asdict
from pathlib import Path
from typing import Any, Dict, List, Tuple

pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above


class CorpusIntegrityError(RuntimeError):
    pass


def _sha(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_corpus(run_dir: Path, spec: DataSpec,
                 blueprints: List[Blueprint],
                 documents: Dict[str, str],
                 report: RenderReport,
                 backend_name: str = "unknown") -> Path:
    run_dir = Path(run_dir)
    (run_dir / "docs").mkdir(parents=True, exist_ok=True)
    (run_dir / "truth").mkdir(parents=True, exist_ok=True)
    hashes: Dict[str, str] = {}
    for bp in blueprints:
        text = documents.get(bp.doc_id, "")
        doc_rel = "docs/{}.txt".format(bp.doc_id)
        truth_rel = "truth/{}.json".format(bp.doc_id)
        (run_dir / doc_rel).write_text(text, encoding="utf-8")
        truth_json = bp.to_json()
        (run_dir / truth_rel).write_text(truth_json, encoding="utf-8")
        hashes[doc_rel] = _sha(text)
        hashes[truth_rel] = _sha(truth_json)
    report_json = json.dumps(asdict(report), indent=2)
    (run_dir / "render_report.json").write_text(report_json,
                                                encoding="utf-8")
    manifest = {
        "synthkit_manifest": 1,
        "spec": json.loads(spec.to_json()),
        "master_seed": spec.corpus.master_seed,
        "backend": backend_name,
        "documents": len(blueprints),
        "render": {
            "first_try": report.first_try,
            "retried": report.retried,
            "fallbacks": report.fallbacks,
        },
        "hashes": hashes,
    }
    (run_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8")
    return run_dir


def load_corpus(run_dir: Path,
                ) -> Tuple[DataSpec, List[Blueprint],
                           Dict[str, str], Dict[str, Any]]:
    """Returns (spec, blueprints, documents, manifest). Every hash
    verified; any mismatch raises CorpusIntegrityError naming the
    file."""
    run_dir = Path(run_dir)
    manifest = json.loads(
        (run_dir / "manifest.json").read_text(encoding="utf-8"))
    spec = DataSpec.from_json(json.dumps(manifest["spec"]))
    documents: Dict[str, str] = {}
    blueprints: List[Blueprint] = []
    bad: List[str] = []
    for rel, expected in sorted(manifest["hashes"].items()):
        path = run_dir / rel
        text = path.read_text(encoding="utf-8")
        if _sha(text) != expected:
            bad.append(rel)
            continue
        if rel.startswith("docs/"):
            documents[path.stem] = text
        else:
            raw = json.loads(text)
            blueprints.append(_blueprint_from_raw(raw))
    if bad:
        raise CorpusIntegrityError(
            "corpus integrity failed for: {}".format(", ".join(bad)))
    blueprints.sort(key=lambda b: b.doc_index)
    return spec, blueprints, documents, manifest


def _blueprint_from_raw(raw: Dict[str, Any]) -> Blueprint:
    pass  # intra-package import inlined above
    return Blueprint(
        doc_id=raw["doc_id"],
        doc_index=raw["doc_index"],
        seed=raw["seed"],
        structured=raw.get("structured", {}),
        notes=[
            PlannedNote(
                field_name=n["field_name"],
                note_type=n["note_type"],
                style=n["style"],
                length_words=n["length_words"],
                elements=[PlannedElement(**e)
                          for e in n.get("elements", [])],
                distractors=[PlannedDistractor(**d)
                             for d in n.get("distractors", [])],
            )
            for n in raw.get("notes", [])
        ],
    )


## Library: The Evaluator — exact, sliced scoring against planted truth

*Source of truth: `synthkit/evaluator.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S4: the Evaluator — outside model vs blueprint truth.

Because synthkit PLANTED every fact, evaluation is exact alignment,
not judgment: an extraction matches a blueprint element when the
planted VALUE appears in the extraction's text (normalized), and —
when the rule demands it — the extraction's category agrees. The
same mechanism makes distractors provable false positives: if the
model reports the discontinued medication's value under a
current-medication category, that is a counted FP, not an opinion.

Scoring:
  per element    planted / found / missed -> recall
  distractors    planted / falsely extracted -> fp_rate
  slices         recall by difficulty and by every style axis the
                 planner controlled (persona, verbosity,
                 abbreviation) — the report that says WHERE a model
                 breaks, which no real-data test can isolate.
  unmatched      extractions matching nothing are reported but not
                 penalized (models may extract beyond the spec).

The evaluator is pure: (blueprints, documents, extractions) in,
EvalReport out. Adapters supply the extractions; the bundled
FunctionExtractor wraps any callable for quick harnessing.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass, field
from typing import Any, Callable, Dict, List, Optional

pass  # intra-package import inlined above


# ===================================================================
# Extraction side
# ===================================================================

@dataclass
class Extraction:
    """One thing the outside model claims to have extracted.

    category  the model's own label for it ("current_medication",
              "medication", "allergy", ...) — free text, matched by
              substring against rule categories
    text      the extracted content
    """
    category: str
    text: str


class ExtractorAdapter:
    """Protocol: extract(doc_id, doc_text) -> List[Extraction].
    Wrap the outside model here (HTTP call, SDK, subprocess — the
    evaluator does not care)."""

    name = "base"

    def extract(self, doc_id: str, doc_text: str) -> List[Extraction]:
        raise NotImplementedError


class FunctionExtractor(ExtractorAdapter):
    """Adapter for any callable(doc_id, doc_text) -> [Extraction]."""

    def __init__(self, fn: Callable[[str, str], List[Extraction]],
                 name: str = "function"):
        self._fn = fn
        self.name = name

    def extract(self, doc_id: str, doc_text: str) -> List[Extraction]:
        return self._fn(doc_id, doc_text)


# ===================================================================
# Matching rules
# ===================================================================

@dataclass
class MatchRule:
    """How a planted element (or distractor) aligns with extractions.

    mode         "value"  — planted value appears in extraction text
                 "phrase" — planted phrasing appears (for elements
                            with no {value})
    categories   optional list of substrings; when given, only
                 extractions whose category contains one (case-
                 insensitive) can match. This is what turns a
                 distractor hit into a PROVABLE false positive: the
                 discontinued med extracted under a current-med
                 category.
    """
    mode: str = "value"
    categories: Optional[List[str]] = None


def _norm(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip().lower()


def _category_ok(rule: MatchRule, category: str) -> bool:
    if not rule.categories:
        return True
    cat = _norm(category)
    return any(_norm(c) in cat for c in rule.categories)


def _needle_for(rule: MatchRule, value: Optional[str],
                phrasing: str) -> str:
    if rule.mode == "phrase" or not value:
        return _norm(phrasing)
    return _norm(value)


# ===================================================================
# Report
# ===================================================================

@dataclass
class ElementScore:
    element_id: str
    planted: int = 0
    found: int = 0
    missed_docs: List[str] = field(default_factory=list)

    @property
    def recall(self) -> float:
        return self.found / self.planted if self.planted else 0.0


@dataclass
class DistractorScore:
    distractor_id: str
    planted: int = 0
    false_positives: int = 0
    fp_docs: List[str] = field(default_factory=list)

    @property
    def fp_rate(self) -> float:
        return (self.false_positives / self.planted
                if self.planted else 0.0)


@dataclass
class EvalReport:
    extractor_name: str
    documents: int
    elements: Dict[str, ElementScore]
    distractors: Dict[str, DistractorScore]
    recall_by_difficulty: Dict[str, Dict[str, float]]
    recall_by_style: Dict[str, Dict[str, Dict[str, float]]]
    unmatched_extractions: int
    total_extractions: int

    @property
    def overall_recall(self) -> float:
        planted = sum(e.planted for e in self.elements.values())
        found = sum(e.found for e in self.elements.values())
        return found / planted if planted else 0.0

    def to_json(self) -> str:
        raw = asdict(self)
        raw["overall_recall"] = round(self.overall_recall, 4)
        for eid, e in self.elements.items():
            raw["elements"][eid]["recall"] = round(
                self.elements[eid].recall, 4)
        for did in self.distractors:
            raw["distractors"][did]["fp_rate"] = round(
                self.distractors[did].fp_rate, 4)
        return json.dumps(raw, indent=2, ensure_ascii=False)

    def format_text(self) -> str:
        lines = [
            "EVALUATION: {} on {} document(s)".format(
                self.extractor_name, self.documents),
            "overall element recall: {:.1%}".format(
                self.overall_recall),
            "",
            "PER ELEMENT:",
        ]
        for eid, e in sorted(self.elements.items()):
            lines.append(
                "  {:<28} recall {:>6.1%}  ({}/{} planted{})".format(
                    eid, e.recall, e.found, e.planted,
                    "; missed: " + ", ".join(e.missed_docs[:4])
                    + ("..." if len(e.missed_docs) > 4 else "")
                    if e.missed_docs else "",
                ))
        if self.distractors:
            lines.append("")
            lines.append("DISTRACTORS (false-positive traps):")
            for did, d in sorted(self.distractors.items()):
                lines.append(
                    "  {:<28} fp rate {:>5.1%}  ({}/{} planted)".format(
                        did, d.fp_rate, d.false_positives, d.planted))
        lines.append("")
        lines.append("RECALL BY DIFFICULTY:")
        for diff, cell in sorted(self.recall_by_difficulty.items()):
            lines.append("  {:<8} {:>6.1%}  ({} planted)".format(
                diff, cell["recall"], int(cell["planted"])))
        for axis, values in sorted(self.recall_by_style.items()):
            lines.append("")
            lines.append("RECALL BY {}:".format(axis.upper()))
            for val, cell in sorted(values.items()):
                lines.append("  {:<12} {:>6.1%}  ({} planted)".format(
                    val, cell["recall"], int(cell["planted"])))
        lines.append("")
        lines.append("extractions: {} total, {} matched nothing "
                     "(reported, not penalized)".format(
                         self.total_extractions,
                         self.unmatched_extractions))
        return "\n".join(lines)


# ===================================================================
# The evaluation
# ===================================================================

def evaluate(
    blueprints: List[Blueprint],
    documents: Dict[str, str],
    extractor: ExtractorAdapter,
    rules: Optional[Dict[str, MatchRule]] = None,
) -> EvalReport:
    """Run the extractor over every document and align against the
    blueprints. `rules` maps element_id/distractor_id -> MatchRule
    (missing ids get the default value-match-any-category rule)."""
    rules = rules or {}

    def rule_for(key: str) -> MatchRule:
        return rules.get(key, MatchRule())

    elements: Dict[str, ElementScore] = {}
    distractors: Dict[str, DistractorScore] = {}
    diff_cells: Dict[str, List[int]] = {}
    style_cells: Dict[str, Dict[str, List[int]]] = {}
    unmatched = 0
    total = 0

    for bp in blueprints:
        text = documents.get(bp.doc_id)
        if text is None:
            continue
        extractions = extractor.extract(bp.doc_id, text)
        total += len(extractions)
        used = [False] * len(extractions)

        for note in bp.notes:
            for el in note.elements:
                score = elements.setdefault(
                    el.element_id, ElementScore(el.element_id))
                score.planted += 1
                rule = rule_for(el.element_id)
                needle = _needle_for(rule, el.value, el.phrasing)
                hit = False
                for i, ex in enumerate(extractions):
                    if not _category_ok(rule, ex.category):
                        continue
                    if needle and needle in _norm(ex.text):
                        hit = True
                        used[i] = True
                        break
                if hit:
                    score.found += 1
                else:
                    score.missed_docs.append(bp.doc_id)
                dc = diff_cells.setdefault(el.difficulty, [0, 0])
                dc[0] += 1
                dc[1] += 1 if hit else 0
                for axis, val in note.style.items():
                    ax = style_cells.setdefault(axis, {})
                    cell = ax.setdefault(val, [0, 0])
                    cell[0] += 1
                    cell[1] += 1 if hit else 0

            for dtr in note.distractors:
                dscore = distractors.setdefault(
                    dtr.distractor_id,
                    DistractorScore(dtr.distractor_id))
                dscore.planted += 1
                rule = rule_for(dtr.distractor_id)
                needle = _needle_for(rule, dtr.value, dtr.phrasing)
                for i, ex in enumerate(extractions):
                    if not _category_ok(rule, ex.category):
                        continue
                    if needle and needle in _norm(ex.text):
                        dscore.false_positives += 1
                        dscore.fp_docs.append(bp.doc_id)
                        used[i] = True
                        break

        unmatched += sum(1 for u in used if not u)

    return EvalReport(
        extractor_name=extractor.name,
        documents=len([b for b in blueprints
                       if b.doc_id in documents]),
        elements=elements,
        distractors=distractors,
        recall_by_difficulty={
            k: {"planted": v[0],
                "recall": (v[1] / v[0]) if v[0] else 0.0}
            for k, v in diff_cells.items()
        },
        recall_by_style={
            axis: {
                val: {"planted": c[0],
                      "recall": (c[1] / c[0]) if c[0] else 0.0}
                for val, c in vals.items()
            }
            for axis, vals in style_cells.items()
        },
        unmatched_extractions=unmatched,
        total_extractions=total,
    )


## Library: The Harness — hypotheses become measured experiments

*Source of truth: `synthkit/harness.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S5: the harness — hypotheses become experiments.

The rest of synthkit answers "how does model X behave on data with
properties Y?" The harness turns that into a verdict machine:

    Experiment = spec + extractor + conditions
    run_experiment -> ExperimentResult(passed, measured, finding)

A Condition is a metric path against a threshold —
    overall_recall >= 0.9
    elements.allergy_flag.recall >= 0.85
    distractors.discontinued_medication.fp_rate <= 0.05
    recall_by_style.verbosity.terse.recall >= 0.7
— resolved against the S4 report. All conditions must hold for the
experiment to pass, and the finding names every measurement either
way: the verdict carries its evidence.

This is the piece that plugs synthkit into any hypothesis-testing
loop (Mnemo's forge included, via a thin adapter on that side):
empirical measurement replacing LLM-judged opinion.

StubBackend renders deterministically (planted content, boring
prose) for dry runs and CI; swap in OllamaBackend for realistic
prose without touching the experiment.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional

pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above

OPS = (">=", "<=", ">", "<", "==")


class StubBackend(LLMBackend):
    """Deterministic renderer backend: echoes every REQUIRED CONTENT
    item into plain prose. Valid by construction — the dry-run and
    CI workhorse."""

    name = "stub"

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        reqs: List[str] = []
        active = False
        for line in prompt.splitlines():
            if line.startswith("REQUIRED CONTENT"):
                active = True
                continue
            if active:
                if line.startswith("- "):
                    reqs.append(line[2:])
                elif not line.strip():
                    active = False
        return ("Documented at the bedside today. "
                + " Also noted, ".join(r.rstrip(".") for r in reqs)
                + ". Plan continues as discussed.")


@dataclass
class Condition:
    metric: str          # dotted path into the eval report
    op: str              # one of OPS
    threshold: float

    def holds(self, value: float) -> bool:
        if self.op == ">=":
            return value >= self.threshold
        if self.op == "<=":
            return value <= self.threshold
        if self.op == ">":
            return value > self.threshold
        if self.op == "<":
            return value < self.threshold
        return abs(value - self.threshold) < 1e-9

    def describe(self, value: float) -> str:
        return "{} = {:.3f} (required {} {:.3f}) -> {}".format(
            self.metric, value, self.op, self.threshold,
            "HOLDS" if self.holds(value) else "FAILS")

    @staticmethod
    def parse(text: str) -> "Condition":
        """'elements.x.recall >= 0.9' -> Condition."""
        m = re.match(r"\s*([\w.]+)\s*(>=|<=|>|<|==)\s*([\d.]+)\s*$",
                     text)
        if not m:
            raise ValueError("cannot parse condition: {}".format(text))
        return Condition(metric=m.group(1), op=m.group(2),
                         threshold=float(m.group(3)))


class MetricError(KeyError):
    pass


def resolve_metric(report: EvalReport, path: str) -> float:
    """Dotted-path lookup with the report's computed properties."""
    parts = path.split(".")
    try:
        if parts[0] == "overall_recall":
            return report.overall_recall
        if parts[0] == "elements":
            score = report.elements[parts[1]]
            return {"recall": score.recall,
                    "planted": float(score.planted),
                    "found": float(score.found)}[parts[2]]
        if parts[0] == "distractors":
            d = report.distractors[parts[1]]
            return {"fp_rate": d.fp_rate,
                    "planted": float(d.planted),
                    "false_positives": float(d.false_positives)
                    }[parts[2]]
        if parts[0] == "recall_by_difficulty":
            return float(report.recall_by_difficulty[parts[1]]
                         [parts[2]])
        if parts[0] == "recall_by_style":
            return float(report.recall_by_style[parts[1]][parts[2]]
                         [parts[3]])
        if parts[0] == "unmatched_extractions":
            return float(report.unmatched_extractions)
    except (KeyError, IndexError):
        raise MetricError(
            "metric not present in this report: {}".format(path))
    raise MetricError("unknown metric family: {}".format(path))


@dataclass
class Experiment:
    name: str
    spec: DataSpec
    extractor: ExtractorAdapter
    conditions: List[Condition]
    rules: Optional[Dict[str, MatchRule]] = None
    backend: Optional[LLMBackend] = None    # default StubBackend
    render_attempts: int = 3


@dataclass
class ExperimentResult:
    name: str
    passed: bool
    measured: Dict[str, float]
    failed_conditions: List[str]
    eval_report: EvalReport
    render_report: RenderReport
    documents: int

    def finding(self) -> str:
        lines = [
            "Experiment '{}' on {} synthetic document(s) "
            "({} render fallback(s)): {}.".format(
                self.name, self.documents,
                self.render_report.fallbacks,
                "PASSED" if self.passed else "FAILED"),
        ]
        for cond_desc in self.measured_descriptions:
            lines.append("  " + cond_desc)
        return "\n".join(lines)

    measured_descriptions: List[str] = field(default_factory=list)


def run_experiment(exp: Experiment) -> ExperimentResult:
    """Plan -> render (verified) -> evaluate -> verdict. Deterministic
    for a given spec/seed/backend."""
    exp.spec.validate()
    blueprints = plan_corpus(exp.spec)
    backend = exp.backend or StubBackend()
    documents, render_report = render_corpus(
        exp.spec, blueprints, backend,
        attempts=exp.render_attempts)
    report = evaluate(blueprints, documents, exp.extractor,
                      exp.rules)
    measured: Dict[str, float] = {}
    descriptions: List[str] = []
    failed: List[str] = []
    for cond in exp.conditions:
        value = resolve_metric(report, cond.metric)
        measured[cond.metric] = round(value, 4)
        descriptions.append(cond.describe(value))
        if not cond.holds(value):
            failed.append(cond.metric)
    result = ExperimentResult(
        name=exp.name,
        passed=not failed,
        measured=measured,
        failed_conditions=failed,
        eval_report=report,
        render_report=render_report,
        documents=len(blueprints),
    )
    result.measured_descriptions = descriptions
    return result


## Library: Examples — the reference vertical and a naive real extractor

*Source of truth: `synthkit/examples.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 examples: the reference vertical, a real (naive) regex
extractor, and a canned experiment — everything needed to watch the
whole machine turn over.

reference_spec() is the v1 vertical: inpatient progress notes with
current medications, follow-ups, hard-buried allergies, and a
discontinued-medication distractor trap.

regex_extract() is a deliberately naive but REAL extractor — the
kind of thing a vendor demo might hide behind an API. Its flaws are
the point: it greps medication names anywhere (so it falls into the
discontinued trap), it only knows some phrasings (so recall varies
by surface form), and it has no notion of allergy context beyond one
pattern. Running it through the harness produces an honest, sliced
failure report — the product's output, live.

Replace regex_extract with an adapter around any outside model and
nothing else changes.
"""
from __future__ import annotations

import re
from typing import List

pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above

MED_POOL = [
    "metformin 500mg BID",
    "lisinopril 10mg daily",
    "atorvastatin 40mg nightly",
    "levothyroxine 75mcg qAM",
]
STOPPED_POOL = ["ibuprofen", "omeprazole"]
ALLERGY_POOL = ["penicillin", "sulfa", "latex"]
FOLLOWUP_POOL = ["in two weeks", "in one month", "next Tuesday"]


def reference_spec(size: int = 50, master_seed: int = 42) -> DataSpec:
    return DataSpec(
        title="Progress note extraction test corpus",
        structured_fields=[
            StructuredField(
                name="encounter_id", ftype="id",
                distribution=Distribution(
                    kind="sequence",
                    params={"prefix": "ENC-", "start": 10000}),
            ),
            StructuredField(
                name="patient_name", ftype="person_name",
                distribution=Distribution(kind="categorical",
                                          params={"choices": ["x"]}),
            ),
            StructuredField(
                name="admit_date", ftype="date",
                distribution=Distribution(
                    kind="date_range",
                    params={"start": "2026-01-01",
                            "end": "2026-06-30"}),
            ),
            StructuredField(
                name="discharge_date", ftype="date",
                distribution=Distribution(
                    kind="date_range",
                    params={"start": "2026-01-01",
                            "end": "2026-06-30"}),
            ),
            StructuredField(
                name="length_of_stay_days", ftype="int",
                distribution=Distribution(
                    kind="lognormal",
                    params={"mu": 1.2, "sigma": 0.6,
                            "min": 1, "max": 45}),
            ),
        ],
        unstructured_fields=[
            UnstructuredField(
                name="progress_note",
                note_type="inpatient progress note",
                target_elements=[
                    TargetElement(
                        element_id="current_medication",
                        description="an active medication with dose",
                        phrasings=[
                            "continues {value} at current dose",
                            "pt remains on {value}",
                            "{value} — no changes today",
                        ],
                        density=0.9, difficulty="easy",
                        value_source={"choices": list(MED_POOL)},
                    ),
                    TargetElement(
                        element_id="followup_appointment",
                        description="a scheduled follow-up",
                        phrasings=[
                            "follow up scheduled for {value}",
                            "will see clinic again {value}",
                        ],
                        density=0.6, difficulty="medium",
                        value_source={"choices": list(FOLLOWUP_POOL)},
                    ),
                    TargetElement(
                        element_id="allergy_flag",
                        description="a documented allergy",
                        phrasings=[
                            "allergies: {value}",
                            "known {value} allergy noted",
                        ],
                        density=0.4, difficulty="hard",
                        value_source={"choices": list(ALLERGY_POOL)},
                    ),
                ],
                distractors=[
                    Distractor(
                        distractor_id="discontinued_medication",
                        description="a med explicitly STOPPED — must "
                                    "not be extracted as current",
                        phrasings=[
                            "{value} discontinued this admission",
                            "stopped {value} due to side effects",
                        ],
                        density=0.5,
                        value_source={"choices": list(STOPPED_POOL)},
                    ),
                ],
                style=StyleAxes(
                    personas=["attending", "resident", "nurse"],
                ),
                length_words=[60, 180],
            ),
        ],
        cross_field_rules=[
            CrossFieldRule(earlier="admit_date",
                           later="discharge_date",
                           min_delta=1, max_delta=21),
        ],
        corpus=CorpusConfig(size=size, master_seed=master_seed),
    )


def reference_rules() -> dict:
    return {
        "current_medication": MatchRule(
            mode="value", categories=["current", "active med"]),
        "followup_appointment": MatchRule(mode="value"),
        "allergy_flag": MatchRule(mode="value",
                                  categories=["allerg"]),
        "discontinued_medication": MatchRule(
            mode="value", categories=["current", "active med"]),
    }


# ===================================================================
# The naive-but-real regex extractor
# ===================================================================

_MED_RE = re.compile(
    r"\b((?:metformin|lisinopril|atorvastatin|levothyroxine|"
    r"ibuprofen|omeprazole)[^.\n]*?)(?:[.\n]|$)", re.IGNORECASE)
_FOLLOWUP_RE = re.compile(
    r"follow(?:\s|-)?up (?:scheduled )?for ([^.\n]+)", re.IGNORECASE)
_ALLERGY_RE = re.compile(
    r"allerg(?:y|ies)[:\s]+([^.\n]+)", re.IGNORECASE)


def regex_extract(doc_id: str, text: str) -> List[Extraction]:
    """Deliberately naive: any known med name anywhere becomes a
    'current medication' (the trap), only one follow-up phrasing is
    known, only one allergy pattern is known."""
    out: List[Extraction] = []
    for m in _MED_RE.finditer(text):
        out.append(Extraction(category="current medication",
                              text=m.group(1).strip()))
    for m in _FOLLOWUP_RE.finditer(text):
        out.append(Extraction(category="follow-up",
                              text=m.group(1).strip()))
    for m in _ALLERGY_RE.finditer(text):
        out.append(Extraction(category="allergy",
                              text=m.group(1).strip()))
    return out


def allergy_bar_experiment(backend=None, size: int = 16,
                           extractor_fn=None) -> Experiment:
    """The canned live experiment: does the extractor clear an 85%
    allergy-recall bar and stay under a 10% distractor trap rate?
    (The naive regex extractor does neither — by design.)"""
    pass  # intra-package import inlined above
    return Experiment(
        name="allergy recall bar + discontinued-med trap",
        spec=reference_spec(size=size),
        extractor=FunctionExtractor(extractor_fn or regex_extract,
                                    "regex-naive"),
        conditions=[
            Condition.parse("elements.allergy_flag.recall >= 0.85"),
            Condition.parse(
                "distractors.discontinued_medication.fp_rate <= 0.10"),
        ],
        rules=reference_rules(),
        backend=backend,
    )


---
# Walkthrough

Everything below runs deterministically (StubBackend).
Flip `RUN_BEDROCK = True` for realistic prose (needs
`bedrock:InvokeModel` on the execution role).


In [ ]:
RUN_BEDROCK = False
BEDROCK_MODEL_ID = "anthropic.claude-sonnet-4-6-v1:0"
BEDROCK_REGION = "us-west-2"
CORPUS_DIR = "corpus/notebook_run_001"


## 1. Spec and plan


In [ ]:
spec = reference_spec(size=20, master_seed=42)
spec.validate()
blueprints = plan_corpus(spec)
import json as _json
print(_json.dumps(corpus_stats(blueprints), indent=2))


## 2. Verified render (deterministic)


In [ ]:
documents, render_report = render_corpus(
    spec, blueprints, StubBackend())
print(render_report.format_text())
print()
print(documents["doc_00000"][:400])


## 3. Evaluate the naive extractor

Replace `regex_extract` with an adapter around any real
model: `(doc_id, text) -> [Extraction]`.


In [ ]:
report = evaluate(
    blueprints, documents,
    FunctionExtractor(regex_extract, "regex-naive"),
    reference_rules())
print(report.format_text())


## 4. A measured experiment


In [ ]:
exp = Experiment(
    name="allergy bar + discontinued-med trap",
    spec=spec,
    extractor=FunctionExtractor(regex_extract,
                                "regex-naive"),
    conditions=[
        Condition.parse(
            "elements.allergy_flag.recall >= 0.85"),
        Condition.parse(
            "distractors.discontinued_medication"
            ".fp_rate <= 0.10"),
    ],
    rules=reference_rules(),
)
result = run_experiment(exp)
print(result.finding())


## 5. Persist and reload with integrity


In [ ]:
from pathlib import Path as _P
run_dir = write_corpus(_P(CORPUS_DIR), spec,
                       blueprints, documents,
                       render_report, "stub")
_s2, _b2, _d2, _m = load_corpus(run_dir)
assert _d2 == documents
print("corpus verified ->", run_dir)


## 6. In-notebook open-source LLM (gated)

Generation INSIDE this notebook's process — no server,
no service. First run downloads weights from the
Hugging Face hub (~1-3GB; use a local/S3 path as
`model_id` in air-gapped environments). The 0.5B model
runs on CPU instances; prefer a GPU instance for 1.5B+.
The verifier + retry + fallback wrap it like any
backend — a weak model degrades measurably, never
breaks the corpus.


In [ ]:
RUN_LOCAL_LLM = False
LOCAL_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
if RUN_LOCAL_LLM:
    import sys as _sys
    !{_sys.executable} -m pip install -q transformers torch accelerate
    hf_backend = HFLocalBackend(model_id=LOCAL_MODEL_ID)
    hf_docs, hf_rr = render_corpus(spec, blueprints,
                                   hf_backend)
    print(hf_rr.format_text())
    print(evaluate(
        blueprints, hf_docs,
        FunctionExtractor(regex_extract,
                          "regex-naive"),
        reference_rules()).format_text())
else:
    print("RUN_LOCAL_LLM is False — skipped.")


## 7. Bedrock (gated) — realistic prose


In [ ]:
if RUN_BEDROCK:
    backend = BedrockBackend(
        model_id=BEDROCK_MODEL_ID,
        region=BEDROCK_REGION)
    live_docs, live_rr = render_corpus(
        spec, blueprints, backend)
    print(live_rr.format_text())
    print(evaluate(
        blueprints, live_docs,
        FunctionExtractor(regex_extract,
                          "regex-naive"),
        reference_rules()).format_text())
else:
    print("RUN_BEDROCK is False — skipped.")


## 8. Self-validation

A compact assertion suite over the inlined library —
the notebook proves itself on every full run.


In [ ]:
corpus_b = plan_corpus(reference_spec(size=20,
                                      master_seed=42))
assert all(a.to_json() == b.to_json()
           for a, b in zip(blueprints, corpus_b)), \
    "determinism"
def _perfect(doc_id, text):
    bp = next(b for b in blueprints
              if b.doc_id == doc_id)
    cats = {"current_medication":
            "current medication",
            "followup_appointment": "follow-up",
            "allergy_flag": "allergy"}
    return [Extraction(cats[e.element_id],
                       e.value or e.phrasing)
            for e in bp.notes[0].elements]
_r = evaluate(blueprints, documents,
              FunctionExtractor(_perfect, "perfect"),
              reference_rules())
assert abs(_r.overall_recall - 1.0) < 1e-9, \
    "perfect recall"
assert _r.distractors[
    "discontinued_medication"].false_positives == 0
assert render_report.fallbacks == 0, "verified render"
v = verify_note(documents["doc_00000"],
                blueprints[0].notes[0],
                spec.unstructured_fields[0])
assert v.ok, "verifier"
print("SELF-VALIDATION: all assertions passed.")
